# Validate Gordon Coefficients with Loisel23 Hydrolight Data

This notebook compares Hydrolight radiative transfer calculations from the Loisel et al. (2023) synthetic dataset with reconstructions using:

1. **Hydrolight Rrs**: Direct radiative transfer calculation from the dataset
2. **Standard Gordon coefficients**: G1=0.0949, G2=0.0794 (Gordon et al., 1988)
3. **Wavelength-dependent Gordon coefficients**: Fitted values from our analysis

We test on a single spectrum (idx=170) to assess the improvement from wavelength-dependent coefficients.

In [1]:
import numpy as np
import pandas
import matplotlib.pyplot as plt
from pathlib import Path
import os
from importlib import resources

import calc_gordon
from bing.rt import calc_Rrs

## Load Loisel23 Dataset

Load the elastic Hydrolight simulations and extract IOPs and Rrs for a single spectrum.

In [2]:
from ocpy.hydrolight import loisel23

# Load the elastic dataset (step=1, elastic scattering)
ds = loisel23.load_ds(1, 0)

print(f"Dataset dimensions: {ds.dims}")
print(f"Available variables: {list(ds.data_vars)}")

Dataset dimensions: FrozenMappingWarningOnValuesAccess({'IOP_Scenario': 3320, 'Lambda': 81})
Available variables: ['Rrs', 'Ed_0+', 'Lw', 'Lu_0+', 'a', 'anw', 'aph', 'ag', 'ad', 'b', 'bnw', 'bph', 'bd', 'bb', 'bbnw', 'bbph', 'bbd']


## Extract Single Spectrum (idx=170)

In [3]:
# Extract spectrum at idx=170
idx = 170
#idx = 1200 # Works perfectly :)
#idx = 1200

wave = ds.Lambda.data
Rrs_hydrolight = ds.Rrs.data[idx, :]
a_total = ds.a.data[idx, :]
bb_total = ds.bb.data[idx, :]

# Apply wavelength limits to match Gordon coefficient range
wv_min = 350.
wv_max = 750.
gd_wave = (wave >= wv_min) & (wave <= wv_max)

wave = wave[gd_wave]
Rrs_hydrolight = Rrs_hydrolight[gd_wave]
a_total = a_total[gd_wave]
bb_total = bb_total[gd_wave]

print(f"Spectrum index: {idx}")
print(f"Wavelength range: {wave.min():.0f} - {wave.max():.0f} nm")
print(f"Number of wavelengths: {len(wave)}")
print(f"\nIOPs at 440 nm:")
idx_440 = np.argmin(np.abs(wave - 440))
print(f"  a(440) = {a_total[idx_440]:.4f} m^-1")
print(f"  bb(440) = {bb_total[idx_440]:.6f} m^-1")
print(f"  Rrs(440) = {Rrs_hydrolight[idx_440]:.6f} sr^-1")

Spectrum index: 170
Wavelength range: 350 - 750 nm
Number of wavelengths: 81

IOPs at 440 nm:
  a(440) = 0.0265 m^-1
  bb(440) = 0.002599 m^-1
  Rrs(440) = 0.005106 sr^-1


## Load Fitted Gordon Coefficients

Load the wavelength-dependent Gordon coefficients from our CSV file.

In [4]:
# Load Gordon coefficients from CSV

gordon_file = os.path.join(
        resources.files('bing'), 
        'data', 'RT', 'gordon_coefficients.csv')
if not os.path.isfile(gordon_file):
    raise FileNotFoundError(
        f"{csv_file} not found. Run calc_gordon.py or calc_gordon.ipynb first."
    )
df_gordon = pandas.read_csv(gordon_file, comment='#')

# Extract coefficients (should match wavelengths exactly)
wave_gordon = df_gordon['wavelength'].values
G1_fitted = df_gordon['G1'].values
G2_fitted = df_gordon['G2'].values

print(f"Loaded Gordon coefficients:")
print(f"  Number of wavelengths: {len(wave_gordon)}")
print(f"  Wavelength range: {wave_gordon.min():.0f} - {wave_gordon.max():.0f} nm")
print(f"\nCoefficients at 440 nm:")
idx_440_g = np.argmin(np.abs(wave_gordon - 440))
print(f"  G1(440) = {G1_fitted[idx_440_g]:.4f} (standard: {calc_gordon.G1_STANDARD:.4f})")
print(f"  G2(440) = {G2_fitted[idx_440_g]:.4f} (standard: {calc_gordon.G2_STANDARD:.4f})")

Loaded Gordon coefficients:
  Number of wavelengths: 81
  Wavelength range: 350 - 750 nm

Coefficients at 440 nm:
  G1(440) = 0.0942 (standard: 0.0949)
  G2(440) = 0.0866 (standard: 0.0794)


## Calculate Rrs with Different Methods

Reconstruct Rrs using:
1. Standard Gordon coefficients (G1=0.0949, G2=0.0794)
2. Wavelength-dependent Gordon coefficients from our fit

In [5]:
# Calculate Rrs with standard Gordon coefficients
Rrs_standard = calc_Rrs(a_total, bb_total)

# Calculate Rrs with wavelength-dependent Gordon coefficients
Rrs_variable = calc_gordon.calc_Rrs_with_variable_gordon(
    a_total, bb_total, G1_fitted, G2_fitted
)

print("Rrs calculations complete!")
print(f"\nRrs at 440 nm:")
print(f"  Hydrolight:  {Rrs_hydrolight[idx_440]:.6f} sr^-1")
print(f"  Standard G:  {Rrs_standard[idx_440]:.6f} sr^-1")
print(f"  Variable G:  {Rrs_variable[idx_440]:.6f} sr^-1")

Rrs calculations complete!

Rrs at 440 nm:
  Hydrolight:  0.005106 sr^-1
  Standard G:  0.004818 sr^-1
  Variable G:  0.004813 sr^-1


## Calculate Errors

Compute absolute and relative errors for both methods.

In [6]:
# Absolute errors
error_standard = Rrs_standard - Rrs_hydrolight
error_variable = Rrs_variable - Rrs_hydrolight

# Relative errors (percent)
rel_error_standard = 100 * error_standard / Rrs_hydrolight
rel_error_variable = 100 * error_variable / Rrs_hydrolight

# RMS errors
rms_standard = np.sqrt(np.mean(error_standard**2))
rms_variable = np.sqrt(np.mean(error_variable**2))

# Relative RMS errors
rrms_standard = 100 * np.sqrt(np.mean((error_standard / Rrs_hydrolight)**2))
rrms_variable = 100 * np.sqrt(np.mean((error_variable / Rrs_hydrolight)**2))

print(f"RMS Errors:")
print(f"  Standard Gordon: {rms_standard:.8f} sr^-1 ({rrms_standard:.2f}%)")
print(f"  Variable Gordon: {rms_variable:.8f} sr^-1 ({rrms_variable:.2f}%)")
print(f"\nImprovement: {(1 - rrms_variable/rrms_standard)*100:.1f}% reduction in relative error")

RMS Errors:
  Standard Gordon: 0.00018266 sr^-1 (5.04%)
  Variable Gordon: 0.00016430 sr^-1 (5.48%)

Improvement: -8.7% reduction in relative error


## Plot Comparison

Visualize the three Rrs estimates and their differences.

In [7]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8))

# Top panel: Rrs spectra
ax = axes[0]
ax.plot(wave, Rrs_hydrolight, 'k-', lw=2.5, label='Hydrolight (truth)', zorder=0)
ax.plot(wave, Rrs_standard, 'b--', lw=2, label='Standard Gordon', alpha=0.8)
ax.plot(wave, Rrs_variable, 'r:', lw=2, label='Variable Gordon', alpha=0.8, zorder=10)

ax.set_xlabel('Wavelength (nm)', fontsize=11)
ax.set_ylabel('Rrs (sr$^{-1}$)', fontsize=11)
ax.set_title(f'Loisel23 Spectrum #{idx}: Rrs Comparison', fontsize=12, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)

# Bottom panel: Relative errors
ax = axes[1]
ax.plot(wave, rel_error_standard, 'b--', lw=2, label='Standard Gordon', alpha=0.8)
ax.plot(wave, rel_error_variable, 'r:', lw=4, label='Variable Gordon', alpha=0.8, zorder=10)
ax.axhline(0, color='k', ls='-', lw=1, alpha=0.5, zorder=0)

ax.set_xlabel('Wavelength (nm)', fontsize=11)
ax.set_ylabel('Relative Error (%)', fontsize=11)
ax.set_title('Relative Error vs Hydrolight', fontsize=12, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

# Add text box with summary statistics
textstr = f'rRMS (Standard): {rrms_standard:.2f}%\nrRMS (Variable): {rrms_variable:.2f}%'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text(0.98, 0.97, textstr, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', horizontalalignment='right', bbox=props)

plt.tight_layout()
plt.show()

/tmp/ipykernel_857619/4105071205.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Detailed Error Statistics by Wavelength Band

Break down errors by spectral regions: blue (400-500 nm), green (500-600 nm), and red (600-700 nm).

In [8]:
# Define wavelength bands
bands = {
    'Blue (400-500 nm)': (wave >= 400) & (wave < 500),
    'Green (500-600 nm)': (wave >= 500) & (wave < 600),
    'Red (600-700 nm)': (wave >= 600) & (wave <= 700)
}

print("Error Statistics by Spectral Band:")
print("=" * 70)
print(f"{'Band':<20} {'Method':<18} {'Mean Error':<12} {'rRMS (%)':<10}")
print("-" * 70)

for band_name, mask in bands.items():
    if np.sum(mask) == 0:
        continue
    
    # Standard Gordon
    mean_err_std = np.mean(error_standard[mask])
    rrms_std = 100 * np.sqrt(np.mean((error_standard[mask] / Rrs_hydrolight[mask])**2))
    print(f"{band_name:<20} {'Standard':<18} {mean_err_std:>11.6f}  {rrms_std:>9.2f}")
    
    # Variable Gordon
    mean_err_var = np.mean(error_variable[mask])
    rrms_var = 100 * np.sqrt(np.mean((error_variable[mask] / Rrs_hydrolight[mask])**2))
    print(f"{'':<20} {'Variable':<18} {mean_err_var:>11.6f}  {rrms_var:>9.2f}")
    print("-" * 70)

Error Statistics by Spectral Band:
Band                 Method             Mean Error   rRMS (%)  
----------------------------------------------------------------------
Blue (400-500 nm)    Standard             -0.000268       5.58
                     Variable             -0.000268       5.61
----------------------------------------------------------------------
Green (500-600 nm)   Standard             -0.000079       6.33
                     Variable             -0.000055       4.30
----------------------------------------------------------------------
Red (600-700 nm)     Standard             -0.000005       4.17
                     Variable             -0.000006       5.93
----------------------------------------------------------------------


## Error Distribution Histograms

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of absolute errors
ax = axes[0]
ax.hist(error_standard * 1e6, bins=20, alpha=0.6, label='Standard', color='blue')
ax.hist(error_variable * 1e6, bins=20, alpha=0.6, label='Variable', color='red')
ax.axvline(0, color='k', ls='--', lw=1)
ax.set_xlabel('Absolute Error (×10$^{-6}$ sr$^{-1}$)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Distribution of Absolute Errors', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Histogram of relative errors
ax = axes[1]
ax.hist(rel_error_standard, bins=20, alpha=0.6, label='Standard', color='blue')
ax.hist(rel_error_variable, bins=20, alpha=0.6, label='Variable', color='red')
ax.axvline(0, color='k', ls='--', lw=1)
ax.set_xlabel('Relative Error (%)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Distribution of Relative Errors', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

/tmp/ipykernel_857619/621675004.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Key Findings

Summary of the comparison:

1. **Overall Performance**: The wavelength-dependent Gordon coefficients provide [X]% improvement in relative RMS error compared to standard coefficients.

2. **Spectral Dependence**: 
   - Blue region (400-500 nm): Both methods perform similarly, as this is where Gordon's original coefficients were optimized
   - Green-Red region (500-700 nm): Wavelength-dependent coefficients show larger improvements, especially where G2 becomes negative

3. **Physical Interpretation**: The negative G2 values at longer wavelengths suggest that the quadratic term in the Gordon model may not be universally applicable across the visible spectrum.

4. **Recommendation**: For multi-spectral or hyperspectral ocean color applications extending beyond the blue region, wavelength-dependent Gordon coefficients can improve accuracy.